# 001 · Classes and Objects

> **Syntax › Classes & OOP** · Python 3.14

Python is fully object-oriented — every value, including `int`, functions and classes themselves, is an object. Defining your own classes looks familiar to a C# developer, but there are no field declarations, no `new` keyword, and `self` is spelled out explicitly.

**In this notebook you will learn:**

- the `class` statement and how to create instances (no `new`)
- `__init__` as the "constructor", and where `__new__` fits in
- why `self` is an explicit parameter instead of an implicit `this`
- that instance attributes are created dynamically — and what `__slots__` does about it
- instance methods, `isinstance`, and a first look at `__repr__` / `__str__`

## C# mental model

| C# | Python |
|---|---|
| `class Point { ... }` | `class Point: ...` |
| `public Point(int x, int y) { ... }` | `def __init__(self, x, y): ...` |
| `this.X = x;` | `self.x = x` |
| field / auto-property declarations | none — attributes appear when first assigned |
| `new Point(1, 2)` | `Point(1, 2)` |
| constructor overloads | one `__init__` + default args, or `@classmethod` factories |
| `obj is Point` | `isinstance(obj, Point)` |
| `ToString()` | `__repr__` / `__str__` |

The key mental shift: a Python object is essentially a **dictionary of attributes** attached to a class. The class does not declare a fixed shape; `__init__` simply assigns attributes onto a freshly created object.

## Defining a class and creating instances

```csharp
// C#
public class Point
{
    public int X { get; set; }
    public int Y { get; set; }

    public Point(int x, int y)
    {
        X = x;
        Y = y;
    }
}

var p = new Point(1, 2);
```

In Python the body of the class contains methods; attributes are created inside `__init__`. You create an instance by *calling* the class like a function — there is no `new` keyword.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x          # creates the attribute 'x' on this instance
        self.y = y

p = Point(1, 2)             # no 'new' — just call the class
print(p.x, p.y)
print(type(p))              # <class '__main__.Point'>

Naming conventions (PEP 8): classes use `PascalCase`, methods and attributes use `snake_case`. So C#'s `p.X` usually becomes `p.x` in Python.

The simplest possible class needs a body; `pass` is the "empty block" placeholder:

In [ ]:
class Empty:
    pass

e = Empty()
print(e)          # default repr: <__main__.Empty object at 0x...>

## `__init__` is an initializer, `__new__` is the real constructor

Strictly speaking, `__init__` is not a constructor: by the time it runs, the object **already exists** and is passed in as `self`. Its job is to *initialise* that object. It must return `None`.

Object creation is a two-step process when you call `Point(1, 2)`:

1. `Point.__new__(Point, 1, 2)` — allocates and returns a new, empty instance (the actual constructor)
2. `instance.__init__(1, 2)` — initialises it

You will almost never override `__new__`; it is used for advanced cases like subclassing immutable types (`int`, `str`, `tuple`) or implementing singletons/caching. Use `__init__` for everyday code.

In [ ]:
class Traced:
    def __new__(cls, *args):
        print(f"__new__  called with cls={cls.__name__}, args={args}")
        return super().__new__(cls)       # object.__new__ allocates the instance

    def __init__(self, value):
        print(f"__init__ called with value={value}")
        self.value = value

t = Traced(42)
print(t.value)

## `self` is explicit

In C#, `this` is an implicit, hidden reference available inside every instance method. In Python, the instance is passed as the **first parameter** of every method, and by convention it is named `self`. You must declare it, and you must use it to access attributes — a bare `x` inside a method refers to a local or global variable, **never** to `self.x`.

```csharp
// C#
public double DistanceFromOrigin() => Math.Sqrt(X * X + Y * Y);   // X means this.X
```

In [ ]:
import math

class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def distance_from_origin(self):
        return math.sqrt(self.x ** 2 + self.y ** 2)   # self. is mandatory

    def move_by(self, dx, dy):
        self.x += dx
        self.y += dy

p = Point(3, 4)
print(p.distance_from_origin())   # 5.0
p.move_by(1, 1)
print(p.x, p.y)

Calling `p.distance_from_origin()` is just sugar: Python looks the function up on the class and passes `p` as the first argument. The two calls below are equivalent — which shows that `self` is nothing magical.

In [ ]:
print(p.distance_from_origin())
print(Point.distance_from_origin(p))   # explicit form: the instance passed as 'self'

⚠️ **Gotcha:** forgetting `self` in the parameter list gives a confusing error about argument counts, because Python still passes the instance.

In [ ]:
class Broken:
    def greet():                 # forgot 'self'
        return "hi"

try:
    Broken().greet()
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** forgetting `self.` when reading an attribute is a `NameError` (or silently uses a global with the same name):

In [ ]:
class Counter:
    def __init__(self):
        self.count = 0

    def increment(self):
        count += 1               # meant self.count

try:
    Counter().increment()
except UnboundLocalError as e:   # a subclass of NameError
    print(f"{type(e).__name__}: {e}")

## Attributes are dynamic — no field declarations

C# classes have a fixed shape defined at compile time. A Python instance stores its attributes in a per-instance dictionary, `__dict__`, and **any code can add new attributes at any time**:

In [ ]:
class User:
    def __init__(self, name):
        self.name = name

alice = User("alice")
print(alice.__dict__)            # {'name': 'alice'}

alice.email = "alice@example.com"   # add a brand-new attribute at runtime
print(alice.__dict__)

bob = User("bob")
print(hasattr(bob, "email"))     # False — only 'alice' got it

This flexibility means a typo silently creates a new attribute instead of failing to compile:

In [ ]:
alice.nmae = "Alice"   # typo! No error — a new attribute is created
print(alice.name)      # still 'alice'

Reading a missing attribute raises `AttributeError` (the runtime equivalent of C#'s compile error CS1061). Use `getattr` with a default for optional lookups:

In [ ]:
try:
    bob.email
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

print(getattr(bob, "email", "<no email>"))

**Best practice:** assign *every* attribute in `__init__` (use `None` if there is no value yet), so the shape of the object is visible in one place — just as you would declare all fields at the top of a C# class. Type checkers (mypy, pyright) will then flag typos like `nmae`.

You can also annotate attributes for tooling, which looks closer to C# field declarations:

In [ ]:
class Product:
    def __init__(self, name: str, price: float) -> None:
        self.name: str = name
        self.price: float = price
        self.discount: float | None = None   # declared up-front, no value yet

print(vars(Product("pen", 1.5)))             # vars(obj) == obj.__dict__

## `__slots__` — opting into a fixed shape

If you declare `__slots__`, instances get a fixed set of attribute slots instead of a `__dict__`. That saves memory (useful for millions of small objects) and makes assigning an unknown attribute an error — closer to C# behaviour.

In [ ]:
class SlottedPoint:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

sp = SlottedPoint(1, 2)
try:
    sp.z = 3
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

print(hasattr(sp, "__dict__"))   # False

Most code does not need `__slots__`. Dataclasses can generate it for you with `@dataclass(slots=True)` *(see Classes & OOP › 008 · Dataclasses)*.

## One constructor only — use default arguments (and factories)

Python has no method overloading, so a class has exactly **one** `__init__`. Defining a second one simply replaces the first. Optional and keyword arguments cover most C# overload scenarios:

```csharp
// C#
public Rectangle(double width, double height) { ... }
public Rectangle(double side) : this(side, side) { }
```

In [ ]:
class Rectangle:
    def __init__(self, width, height=None):
        self.width = width
        self.height = width if height is None else height

    def area(self):
        return self.width * self.height

print(Rectangle(2, 3).area())   # 6
print(Rectangle(4).area())      # 16 — a square
print(Rectangle(width=2, height=5).area())

For genuinely different ways to build an object (e.g. `Rectangle.from_string("2x3")`), Python uses **alternative constructors** written with `@classmethod` — the equivalent of C# static factory methods *(see Classes & OOP › 002 · Class and Static Members)*.

## A first look at `__repr__` and `__str__`

The default text of an object (`<__main__.Point object at 0x...>`) is unhelpful. Overriding `ToString()` in C# corresponds to defining `__repr__` (developer-facing, used in the REPL, debugger and containers) and optionally `__str__` (user-facing, used by `print` and f-strings). If only `__repr__` is defined, it is used for both.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

p = Point(1, 2)
print(p)            # uses __repr__ since there is no __str__
print([p, p])       # containers always use __repr__

The full story — along with `__eq__`, operators and more — is in *Classes & OOP › 007 · Dunder Methods*.

## Identity, equality and `isinstance`

Like C# classes (reference types), instances of a plain Python class compare by **identity** by default: two objects with the same attribute values are *not* equal unless you define `__eq__` (or use a dataclass).

In [ ]:
a = Point(1, 2)
b = Point(1, 2)
print(a == b)      # False — default equality is identity (C#: ReferenceEquals)
print(a is a)      # True

Type checks use `isinstance(obj, cls)` — the equivalent of C#'s `obj is Point`. It respects inheritance and accepts a tuple of types. `type(obj) is cls` is an exact-type check (rarely what you want).

```csharp
// C#
if (shape is Point p) { ... }
```

In [ ]:
print(isinstance(a, Point))            # True
print(isinstance(a, object))           # True — everything derives from object
print(isinstance(42, (int, float)))    # True — any of the given types
print(type(a) is Point)                # exact type check

## Classes are objects too

A class is itself an object (of type `type`), so you can store it in a variable, pass it around, and call it later — no reflection or `Activator.CreateInstance` required.

In [ ]:
factory = Point            # just a reference to the class
p = factory(5, 6)
print(p, type(Point), Point.__name__)

## Summary

| Concept | C# | Python |
|---|---|---|
| Define a class | `class Point { }` | `class Point:` |
| Constructor | `public Point(...)` | `def __init__(self, ...)` |
| Allocation hook | (none) | `__new__(cls, ...)` (rarely overridden) |
| Current instance | implicit `this` | explicit `self` parameter |
| Fields | declared in the class | assigned in `__init__` (`self.x = ...`) |
| Create an instance | `new Point(1, 2)` | `Point(1, 2)` |
| Fixed shape | always | opt-in with `__slots__` |
| Overloaded constructors | multiple ctors | default args / `@classmethod` factories |
| Text representation | `ToString()` | `__repr__` / `__str__` |
| Type test | `obj is Point` | `isinstance(obj, Point)` |

## Exercises

1. Write a `BankAccount` class with an `owner` and a `balance` (default `0`), plus `deposit(amount)` and `withdraw(amount)` methods. `withdraw` should raise `ValueError` if the balance would go negative.
2. Add a `__repr__` to `BankAccount` that returns something like `BankAccount(owner='alice', balance=100)`.
3. Create two accounts and explain why `acc1 == acc2` is `False` even when the owner and balance match.
4. Add `__slots__` to `BankAccount` and show that `acc.balanse = 10` (typo) now raises an error.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1, 2, 4
class BankAccount:
    __slots__ = ("owner", "balance")

    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("insufficient funds")
        self.balance -= amount

    def __repr__(self):
        return f"BankAccount(owner={self.owner!r}, balance={self.balance})"

acc = BankAccount("alice", 100)
print(acc)                      # BankAccount(owner='alice', balance=100)

try:
    acc.balanse = 10            # 4 — typo is now an AttributeError
except AttributeError as e:
    print(e)

# 3 — without __eq__, == falls back to identity: two separate objects are
#     never equal, just like two `new` instances of a C# class.
acc1, acc2 = BankAccount("bob", 5), BankAccount("bob", 5)
print(acc1 == acc2)             # False
```

</details>